In [ ]:
# Lab type: review
# Course: AI402 — Retrieval & RAG Systems
# Lesson: Re-Ranking with Cross-Encoders at Production Scale
# Task: A two-stage retrieval service is below. Run it, then answer the
# judgment questions — one design choice nullifies the re-ranker entirely.

# Lab: Reviewing a Re-Ranking Service

**Outputs are cleared.** Run every cell top to bottom. The cross-encoder model (~90 MB) downloads on first use.

## Setup

In [ ]:
!pip install sentence-transformers rank-bm25 faiss-cpu numpy pandas --quiet

In [ ]:
import numpy as np

# The Nimbus Analytics product knowledge base: (doc_id, heading_path, text)
CORPUS = [
    ("plans-overview", "Pricing > Plans",
     "Nimbus Analytics offers three subscription plans: Starter, Teams, and "
     "Enterprise. Starter includes 5 seats and community support. Teams includes "
     "50 seats, shared dashboards, and priority email support. Enterprise includes "
     "unlimited seats, priority support, and advanced security features."),
    ("sso-policy", "Pricing > Enterprise plan",
     "Single sign-on (SSO) with SAML 2.0 is available on the Enterprise plan only. "
     "The Teams plan does not include SSO. Enterprise customers can configure SSO "
     "from the admin console under Security settings."),
    ("seat-pricing", "Pricing > Seats",
     "Per-seat pricing: Starter is $12 per seat per month, Teams is $29 per seat "
     "per month, and Enterprise pricing is custom. Annual billing gives a 20 "
     "percent discount on all plans."),
    ("refund-policy", "Billing > Refunds",
     "Customers can request a full refund within 30 days of purchase. To get your "
     "money back after 30 days, contact billing support; partial refunds are "
     "prorated for annual subscriptions."),
    ("error-e4022", "Troubleshooting > Error codes",
     "Error E4022 means the API rate limit was exceeded. The Starter plan allows "
     "100 requests per minute, Teams 1,000, and Enterprise 10,000. Wait 60 seconds "
     "and retry, or upgrade the plan."),
    ("error-e5001", "Troubleshooting > Error codes",
     "Error E5001 indicates an expired API token. Rotate the token from the admin "
     "console under API settings. Tokens expire after 90 days by default."),
    ("api-export", "API > Export",
     "The export endpoint POST /v2/export creates a CSV export of dashboard data. "
     "Exports are limited to 100,000 rows on Teams and 1 million rows on "
     "Enterprise."),
    ("data-retention", "Security > Data retention",
     "Event data is retained for 13 months on all plans. Enterprise customers can "
     "configure custom retention windows up to 5 years from the admin console."),
    ("priority-support", "Support > Tiers",
     "Priority support with a 4-hour response SLA is included in Teams and "
     "Enterprise plans. Starter includes community support only."),
    ("dashboard-sharing", "Product > Dashboards",
     "Shared dashboards let teammates view and edit the same dashboard. Sharing "
     "outside your workspace requires a public link, available on Teams and "
     "Enterprise."),
    ("audit-logs", "Security > Audit logs",
     "Audit logs record sign-ins, permission changes, and data exports. Audit "
     "logs are an Enterprise-only feature and are retained for 2 years."),
    ("cancel-downgrade", "Billing > Cancellation",
     "You can cancel or downgrade at any time from the billing page. Downgrades "
     "take effect at the end of the current billing period."),
]
DOC_IDS = [d[0] for d in CORPUS]
DOC_TEXTS = [f"{d[1]}: {d[2]}" for d in CORPUS]

# Labelled evaluation queries: (query, set of relevant doc_ids)
EVAL_SET = [
    ("does the teams plan include sso", {"sso-policy"}),
    ("how do I get my money back", {"refund-policy"}),
    ("what does error E4022 mean", {"error-e4022"}),
    ("how long is event data kept", {"data-retention"}),
    ("cost per seat on the teams plan", {"seat-pricing"}),
    ("response time for priority support", {"priority-support"}),
    ("row limit for csv export", {"api-export"}),
    ("rotate an expired api token", {"error-e5001"}),
]
print(f"{len(CORPUS)} documents, {len(EVAL_SET)} labelled queries")

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    return embedder.encode(list(texts), normalize_embeddings=True)

DOC_EMB = embed(DOC_TEXTS)

def dense_search(query, k=5, doc_emb=None, doc_ids=None):
    doc_emb = DOC_EMB if doc_emb is None else doc_emb
    doc_ids = DOC_IDS if doc_ids is None else doc_ids
    scores = doc_emb @ embed([query])[0]
    order = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], float(scores[i])) for i in order]

print(dense_search("does the teams plan include sso", k=3))

## The service under review

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidate_ids, top_k=3):
    texts = {d: t for d, t in zip(DOC_IDS, DOC_TEXTS)}
    pairs = [(query, texts[c]) for c in candidate_ids]
    scores = reranker.predict(pairs)
    order = np.argsort(scores)[::-1][:top_k]
    return [(candidate_ids[i], float(scores[i])) for i in order]

def retrieve_and_rerank(query, top_k=3):
    candidates = [d for d, _ in dense_search(query, k=top_k)]
    return rerank(query, candidates, top_k=top_k)

for q, rel in EVAL_SET[:3]:
    out = retrieve_and_rerank(q)
    print(f"{q!r:45} -> {[d for d, _ in out]}  (relevant: {sorted(rel)})")

**Question 1.** Compare `retrieve_and_rerank`'s output with plain `dense_search` at the same `top_k` across the whole eval set. How often does the re-ranker change *which documents* are returned (not just their order)? Explain why, pointing at one line of code.

<details>
<summary>🔑 Reveal answer — Question 1</summary>

Never. Stage one retrieves `k=top_k` candidates, so the re-ranker can only permute the same three documents — the funnel has no wide end. The offending line is `dense_search(query, k=top_k)`: the candidate set's *composition* is fixed entirely by the weaker bi-encoder, and the cross-encoder's accuracy is spent shuffling it. Re-ranking recovers ranking errors, never recall errors.

</details>

**Question 2.** Fix the funnel: retrieve a wider candidate set (try k=8) and re-rank down to 3. Measure recall@3 before and after on EVAL_SET. What is the ratio between stage-one k and final top_k here, and what does the lesson recommend?

In [ ]:
# Work here: widen stage one, re-rank to top 3, compare recall@3.


<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
def funnel(query, top_k=3, depth=8):
    candidates = [d for d, _ in dense_search(query, k=depth)]
    return [d for d, _ in rerank(query, candidates, top_k=top_k)]

for fn in (lambda q: [d for d, _ in retrieve_and_rerank(q)], funnel):
    hits = sum(1 for q, rel in EVAL_SET if set(fn(q)) & rel)
    print(hits / len(EVAL_SET))
```

Here depth/top_k ≈ 2.7× because the corpus has only 12 documents; the lesson's production guidance is 10–40× (e.g. 50–200 candidates for a top-5), with the right depth found where measured ranking quality plateaus.

</details>

**Question 3.** Run the widened funnel on the query `"can I export my dashboards to powerpoint"` (the corpus has no such feature) and print the re-ranker's scores. Compare them with the scores for an answerable query. What capability does this give the pipeline that cosine similarity alone cannot?

<details>
<summary>🔑 Reveal answer — Question 3</summary>

The cross-encoder's scores for the unanswerable query sit far below the scores it gives genuine answers (for this model, well into negative logits), while cosine similarity still dutifully produces a "best" match. A calibrated score floor lets the pipeline return *nothing* and say so — the honest no-answer — instead of feeding known-irrelevant context to generation. Stage one alone cannot tell you this.

</details>

**Question 4.** This service re-ranks with one forward pass per candidate per query. Name the two knobs from the lesson that control re-ranking latency at production traffic, and state what each trades away.

<details>
<summary>🔑 Reveal answer — Question 4</summary>

**Candidate depth** — capping it caps compute linearly but lowers the recall ceiling the re-ranker can exploit; set it where measured quality plateaus. **Model size** — a smaller distilled re-ranker (this 6-layer MiniLM is already the workhorse class) cuts per-pair cost at some accuracy loss. Batching the pairs is free speed with no trade; skipping re-ranking when stage-one scores are already well separated trades a little quality on ambiguous queries for a large average saving.

</details>

## Summary

1. Stage one owns _______; stage two owns precision at the top.
2. A funnel whose stage-one k equals its final top_k gives the re-ranker _______ to do.
3. A calibrated cross-encoder score floor enables the honest _______.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **recall**
2. **nothing (except reordering the same set)**
3. **no-answer** — returning no context when nothing is relevant.

</details>